# N158 · Pandas连接、聚合与宽长变换

**目标：** 明确连接键、分组键和输出形状。

**先修：** N157, N154。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** groupby/agg/transform；merge；concat；pivot/pivot_table；melt；多对多。

## 从问题到算法

连接不一定保持行数：一对多键连接会复制左表行，多对多甚至会产生笛卡尔式膨胀。因此先明确键的唯一性，再判断应使用 `concat`、`merge`、`pivot` 还是集合式反连接。

纵向拼表要求相同列序；宽表到长表把“季度列”变成“季度值”，四季度的 n 行变成 4n 行。`pivot` 不是聚合，重复 `(city, month)` 没有唯一温度，应报错而非擅自平均。

未下单客户用成员关系取反。右表同一客户重复下单不应复制结果，缺失的订单客户键也不能导致全部结果消失。

## 最小实现

**本章接口：** `concatenate_tables(df1, df2)`、`pivot_weather(weather)`、`melt_sales(report)`、`customers_without_orders(customers, orders)`

In [1]:
import pandas as pd
from pandas.testing import assert_frame_equal

def concatenate_tables(df1, df2):
    if list(df1.columns) != list(df2.columns):
        raise ValueError('Both tables must have the same ordered schema')
    return pd.concat([df1, df2], ignore_index=True)

def pivot_weather(weather):
    """One measurement per (city, month); duplicates are errors, not averages."""
    if weather.duplicated(['city', 'month']).any():
        raise ValueError('Duplicate (city, month) measurement')
    return weather.pivot(index='month', columns='city', values='temperature').sort_index().sort_index(axis=1)

def melt_sales(report):
    quarters = ['quarter_1','quarter_2','quarter_3','quarter_4']
    return report.melt(id_vars=['product'], value_vars=quarters,
                       var_name='quarter', value_name='sales').reset_index(drop=True)

def customers_without_orders(customers, orders):
    # NULL customerId does not match a real customer, just as with SQL NOT EXISTS.
    ordered_ids = orders['customerId'].dropna()
    return customers.loc[~customers['id'].isin(ordered_ids), ['name']].rename(
        columns={'name':'Customers'}).reset_index(drop=True)

## 正确性、前提与复杂度

反连接结果恰是不存在匹配右表键的左表行；重复右表键不改变存在性。宽长变换用 `(product, quarter)` 标识每个原单元格；唯一键前提成立时可以逆变换。

**代价：** 拼接、melt、哈希成员关系通常为 O(n+m)；四季度只是固定因子。pivot 的组织与排序可达 O(n log n)，存储量取决于展开后的行列组合，稀疏输入也可能产生较大的宽表。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

sales = pd.DataFrame({'product':['A','B'], 'quarter_1':[1,5], 'quarter_2':[2,6], 'quarter_3':[3,7], 'quarter_4':[4,8]})
long = melt_sales(sales)
show_table(['representation','rows','columns'], [['wide',len(sales),list(sales.columns)],['long',len(long),list(long.columns)]])
display(long)
weather = pd.DataFrame({'city':['Seoul','Paris','Seoul','Paris'], 'month':['Jan','Jan','Feb','Feb'], 'temperature':[0,5,2,7]})
display(pivot_weather(weather))

representation,rows,columns
wide,2,"['product', 'quarter_1', 'quarter_2', 'quarter_3', 'quarter_4']"
long,8,"['product', 'quarter', 'sales']"


,product,quarter,sales
0,A,quarter_1,1
1,B,quarter_1,5
2,A,quarter_2,2
3,B,quarter_2,6
4,A,quarter_3,3
5,B,quarter_3,7
6,A,quarter_4,4
7,B,quarter_4,8


city,Paris,Seoul
month,,
Feb,7,2
Jan,5,0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
a = pd.DataFrame({'x':[1,2], 'y':['a','b']})
b = pd.DataFrame({'x':[3], 'y':['c']})
assert_frame_equal(concatenate_tables(a,b),pd.DataFrame({'x':[1,2,3],'y':['a','b','c']}))
assert_frame_equal(concatenate_tables(a.iloc[:0],a),a)
try:
    concatenate_tables(a,b[['y','x']])
except ValueError:
    pass
else:
    raise AssertionError('Schema mismatch was accepted')
r = pd.DataFrame({'product':['A','B'], 'quarter_1':[1,5], 'quarter_2':[2,6], 'quarter_3':[3,7], 'quarter_4':[4,8]})
expected = pd.DataFrame({'product':['A','B']*4,
 'quarter':[q for q in ['quarter_1','quarter_2','quarter_3','quarter_4'] for _ in range(2)],
 'sales':[1,5,2,6,3,7,4,8]})
assert_frame_equal(melt_sales(r),expected)
assert len(melt_sales(r)) == 4*len(r)
w = pd.DataFrame({'city':['A','B','A','B'], 'month':[1,1,2,2], 'temperature':[10,20,11,21]})
expected = pd.DataFrame({'A':[10,11], 'B':[20,21]},index=pd.Index([1,2],name='month'))
expected.columns.name = 'city'
assert_frame_equal(pivot_weather(w),expected)
try:
    pivot_weather(pd.concat([w,w.iloc[:1]]))
except ValueError:
    pass
else:
    raise AssertionError('Ambiguous pivot was accepted')
customers = pd.DataFrame({'id':[1,2,3], 'name':['a','b','c']})
orders = pd.DataFrame({'customerId':[1,1,None]})
assert_frame_equal(customers_without_orders(customers,orders),pd.DataFrame({'Customers':['b','c']}))
import sqlite3
with sqlite3.connect(':memory:') as conn:
    customers.to_sql('Customers',conn,index=False)
    orders.to_sql('Orders',conn,index=False)
    sql = pd.read_sql_query('SELECT name AS Customers FROM Customers c WHERE NOT EXISTS (SELECT 1 FROM Orders o WHERE o.customerId=c.id) ORDER BY c.id',conn)
assert_frame_equal(customers_without_orders(customers,orders),sql)
print('N158: 所有本章断言通过')

N158: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造重复键使pivot报错并解释应否聚合。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** SQL反连接与Pandas结果互验。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 2888. Reshape Data: Concatenate（canonical）：[官方入口](https://leetcode.com/problems/reshape-data-concatenate/)
- 2889. Reshape Data: Pivot（canonical）：[官方入口](https://leetcode.com/problems/reshape-data-pivot/)
- 2890. Reshape Data: Melt（canonical）：[官方入口](https://leetcode.com/problems/reshape-data-melt/)
- 183. Customers Who Never Order（comparison）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。